To force a PySpark notebook to create a new Spark session with custom configurations, rather than using an existing or default one, follow these steps:

- **Stop any existing Spark Session:** If a Spark session is already active in your notebook environment (e.g., from a previous run or automatic startup), you must stop it before creating a new one. This ensures that the new session is truly independent and applies your custom configurations from scratch.

In [1]:
if 'spark' in locals() and spark is not None:
    spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

spark = SparkSession.builder \
      .appName("medals_matches_homework") \
      .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
      .getOrCreate()

spark

25/09/11 20:12:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
SHARED_PATH = "/home/iceberg/notebooks/notebooks/data"

match_details = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/match_details.csv")
matches = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/matches.csv")
medals_matches_players = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/medals_matches_players.csv")
medals = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/medals.csv")
maps = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/maps.csv")

print(f"{match_details.count()}, {matches.count()}, {medals_matches_players.count()}, {medals.count()}, {maps.count()}")

151761, 24025, 755229, 183, 40


In [4]:
medals_joined = medals_matches_players.join(broadcast(medals), "medal_id")

medals_joined.head()

Row(medal_id=3261908037, match_id='009fdac5-e15c-47c6-a202-e18ff8800ce7', player_gamertag='EcZachly', count=7, sprite_uri='https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png', sprite_left=375, sprite_top=525, sprite_sheet_width=74, sprite_sheet_height=74, sprite_width=1125, sprite_height=899, classification='WeaponProficiency', description='Kill an opponent by shooting them in the head.', name='Headshot', difficulty=60)

In [5]:
matches_joined = matches.join(broadcast(maps), "mapid")

matches_joined.head()

Row(mapid='c7edbf0f-f206-11e4-aa52-24be05e24f7e', match_id='11de1a94-8d07-4162-9f5f-d3cc753c811c', is_team_game=True, playlist_id='f72e0ef0-7c4a-4307-af78-8e38dac3fdba', game_variant_id='1e473914-46e4-408d-af26-178fb115de76', is_match_over=True, completion_date=datetime.datetime(2016, 2, 22, 0, 0), match_duration=None, game_mode=None, map_variant_id=None, name='Breakout Arena', description='The broadcast of Breakout matches has proven immensely popular with the UNSC Infinity crew.')

In [6]:
spark.sql("DROP TABLE IF EXISTS bootcamp.bucketed_match_details")
ddl_match_details = """
CREATE TABLE IF NOT EXISTS bootcamp.bucketed_match_details (
     match_id STRING,
     player_gamertag STRING,
     player_total_kills INTEGER,
     player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(ddl_match_details)

spark.sql("DROP TABLE IF EXISTS bootcamp.bucketed_matches")
ddl_matches = """
CREATE TABLE IF NOT EXISTS bootcamp.bucketed_matches (
     match_id STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
USING iceberg
PARTITIONED BY (completion_date, bucket(16, match_id));
"""
spark.sql(ddl_matches)

spark.sql("DROP TABLE IF EXISTS bootcamp.bucketed_medal_matches_players")
ddl_match_details = """
CREATE TABLE IF NOT EXISTS bootcamp.bucketed_medal_matches_players (
     match_id STRING,
     player_gamertag STRING,
     medal_id STRING,
     count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(ddl_match_details)

DataFrame[]

In [8]:
# Saving DataFrames as bucketed tables
match_details.select("match_id", "player_gamertag", "player_total_kills", "player_total_deaths") \
    .write.mode("overwrite") \
    .bucketBy(16, "match_id") \
    .saveAsTable("bootcamp.bucketed_match_details")

matches.select("match_id", "is_team_game", "playlist_id", "completion_date") \
    .write.mode("overwrite") \
    .partitionBy("completion_date") \
    .bucketBy(16, "match_id") \
    .saveAsTable("bootcamp.bucketed_matches")

medals_matches_players.write.mode("overwrite") \
    .bucketBy(16, "match_id") \
    .saveAsTable("bootcamp.bucketed_medal_matches_players")

# Reading the bucketed tables
bucketed_match_details = spark.table("bootcamp.bucketed_match_details")
bucketed_matches = spark.table("bootcamp.bucketed_matches")
bucketed_medal_matches_players = spark.table("bootcamp.bucketed_medal_matches_players")